In [1]:
import os
import numpy as np
import pandas as pd
import math
import datetime as dt
from scipy import stats
import numpy.ma as ma
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from matplotlib.offsetbox import AnchoredText
from math import sin, cos, sqrt, atan2, radians
from collections import Counter
import import_ipynb

# Import functions needed to obtain FOFEM inputs
from functions_calc_hourly_growth import relative_frac
from functions import get_hourly_fuelmois

import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_pacf
import xarray as xr
import geopandas as gpd
from scipy.spatial import cKDTree
from shapely.geometry import Point, Polygon, box
import rasterio
from rasterio.mask import mask
import ast
import re
import matplotlib.patches as mpatches
from netCDF4 import Dataset
import wrf
from wrf import (to_np, getvar, smooth2d, get_cartopy, cartopy_xlim,
                 cartopy_ylim, latlon_coords, interplevel, xy_to_ll, ll_to_xy, destagger)

In [2]:
### Open hourly HDWI values for selected wildfire(s)
ROOT = Path.cwd().parents[1]
creek_hdwi_path = ROOT / "data" / "MP_2020_creekfire_HDWI_reanalysis.csv"
hdwi_out = pd.read_csv(creek_hdwi_path, sep = ",")

hdwi_out['datetime'] = pd.to_datetime(hdwi_out['datetime'])
hdwi_out['fire_year'] = hdwi_out['datetime'].dt.year

# Get daily Fire Atlas growth
hdwi_out['datetime_day'] = hdwi_out['datetime'].dt.date
hdwi_out['datetime_day'] = pd.to_datetime(hdwi_out['datetime_day'])
hdwi_out = hdwi_out.rename(columns={"daily_growth": "daily_acres_fireatlas"})

In [3]:
### Calculate hourly growth using HDWI scaling method
# Filter for the Creek Fire (test case)
hdwi_creek = hdwi_out[hdwi_out['fire_id'] == '2020-CASNF-001391']
hdwi_creek['ambient_temp'] = hdwi_creek['t2']
hdwi_creek['windspeed'] = hdwi_creek['ws']

hourly_growth = hdwi_creek.groupby(['fire_id','fire_year']).apply(relative_frac).reset_index(drop=True)

# Format datetime column and extract the date
hourly_growth['datetime'] = pd.to_datetime(hourly_growth['datetime'])
hourly_growth['date'] = hourly_growth['datetime'].dt.date

Skipping HDWI profile modification


/uufs/chpc.utah.edu/common/home/u6043241/fire_atlas/run_fofem/paper_demo/src/get_fofem_inputs/calc_hourly_growth.py:131: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  modified_hdwi['hourly_growth'][0:3] = np.logspace(0.01, np.log(next_growth

In [4]:
### Add fuel moisture, fuel class, and other input variables needed for FOFEM
hourly_growth = hourly_growth.reset_index(drop=True, inplace=False)
print(hourly_growth)
hourly_growth = get_hourly_fuelmois(hourly_growth)
print(hourly_growth)

hourly_growth['ambient_temp'] = hourly_growth['ambient_temp'] - 273.15
hourly_growth['datetime'] = pd.to_datetime(hourly_growth['datetime'])
hourly_growth['datetime'] = hourly_growth['datetime'].dt.strftime('%Y%m%d%H')

### FOFEM constants
hourly_growth['timestep'] = 15
hourly_growth['percent_crown_burn'] = 50
hourly_growth['emission_factors'] = 'urbanski'
hourly_growth['fire_type'] = 'wildfire'
hourly_growth['fuelcat'] = 'Natural'
hourly_growth['dufftype'] = 'DuffLow'
hourly_growth['met_dat_path'] = 'FALSE'
hourly_growth['con_ob'] = 'FALSE'
fofem_inputs = hourly_growth.drop(columns=['HDWI','max_hdwi_index','sum_hourly_growth', '1h_FM', '10h_FM', '100h_FM', '1000h_FM','daily_acres_fireatlas', 'relative_frac','year','month','date'])

                datetime        lat         lon        HDWI  ambient_temp  \
0    2020-09-04 21:00:00  37.219269 -119.310262  209.533500    303.897733   
1    2020-09-04 22:00:00  37.219269 -119.310262  191.898010    304.263787   
2    2020-09-04 23:00:00  37.219269 -119.310262  166.367230    304.075500   
3    2020-09-05 00:00:00  37.331963 -119.288797  329.139000    302.642681   
4    2020-09-05 01:00:00  37.331963 -119.288797  277.406500    300.986654   
...                  ...        ...         ...         ...           ...   
1510 2020-11-06 19:00:00  37.584184 -119.326361   89.595460    281.560405   
1511 2020-11-06 20:00:00  37.584184 -119.326361   91.423860    281.042407   
1512 2020-11-06 21:00:00  37.584184 -119.326361   83.386086    280.059717   
1513 2020-11-06 22:00:00  37.584184 -119.326361   72.429760    278.583576   
1514 2020-11-06 23:00:00  37.584184 -119.326361   54.858986    276.862493   

      windspeed  num_fccs_cells  \
0      3.028442           39799   
1    

In [5]:
### Set these columns to False, specifically as a boolean datatype
fofem_inputs['con_ob'] = 'FALSE'
fofem_inputs['met_dat_path'] = 'FALSE'

fofem_inputs['con_ob'] = fofem_inputs['con_ob'].map({'TRUE': True, 'FALSE': False})

# Set duff and litter fuel moisture
fofem_inputs['fmoisduff'] = fofem_inputs['fmois1kh']
fofem_inputs['fmoislit'] = 0.05  # default

# Clean up the dataframe, remove NAs
fofem_inputs = fofem_inputs.reset_index(drop=True, inplace=False)
fofem_inputs.dropna(how='any', inplace=True)

# Set fuel moisture rows with values less than 0.01 (mininum threshold for fofem) to 0.01
fofem_inputs[['fmois1h', 'fmois10h', 'fmois100h', 'fmois1kh','fmoisduff']] = (
    fofem_inputs[['fmois1h', 'fmois10h', 'fmois100h', 'fmois1kh','fmoisduff']].clip(lower=0.01)
)

In [6]:
### Expand the FCCS classes and their counts which are currently in dictionaries within each row, want rows for each unique fuel class
# Convert each dictionary row into a list of (FCCS_ID, count) tuples
fofem_inputs['FCCS_IDs'] = fofem_inputs['FCCS_IDs'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
fofem_inputs['class_list'] = fofem_inputs['FCCS_IDs'].apply(lambda d: list(d.items()))

# Explode this list into separate rows
fofem_inputs = fofem_inputs.explode('class_list').reset_index(drop=True)

# Extract FCCS_ID and count from the tuple
fofem_inputs = fofem_inputs.dropna(subset=['class_list'])
fofem_inputs['FCCS_ID'] = fofem_inputs['class_list'].apply(lambda x: x[0])
fofem_inputs['FCCS_counts'] = fofem_inputs['class_list'].apply(lambda x: x[1])

# Calculate percentage of each FCCS class for each day
fofem_inputs['FCCS_class_percent'] = fofem_inputs['FCCS_counts'] / fofem_inputs['num_fccs_cells']
fofem_inputs = fofem_inputs.drop(columns=['class_list','FCCS_counts','FCCS_IDs'])

In [7]:
### Rearrange column order as necessary for FOFEM and rename columns as needed
column_order = [
    "lat",
    "lon",
    "date_time",
    "fuelbed_name",
    "timestep",
    "fmois1kh",
    "fmois100h",
    "fmois10h",
    "fmois1h",
    "fmoislit",
    "fmoisduff",
    "percent_crown_burn",
    "fuelcat",
    "dufftype",
    "met_dat_path",
    "ambient_temp",
    "windspeed",
    "emission_factors",
    "fire_type",
    "con_ob",
    "fire_id",
    "hourly_growth",
    "FCCS_class_percent"
]

fofem_inputs = fofem_inputs.rename(columns = {"FCCS_ID":"fuelbed_name"})
fofem_inputs = fofem_inputs.rename(columns = {"datetime":"date_time"})
fofem_inputs = fofem_inputs[column_order]

In [8]:
### Export FOFEM inputs to CSV file, update to preferred output directory
# fofem_inputs.to_csv('2020_creekfire_FA_fofem_inputs.csv', index = False)